In [1]:
# data preprocessing
from datasets import load_dataset
from transformers import AutoTokenizer

# load IMDB dataset
def load_data_and_tokenize():
    # load IMDB dataset 
    dataset = load_dataset("imdb")
    train_data = dataset["train"]
    test_data = dataset["test"]

    train_data = train_data.shuffle(seed=42).select(range(2048))
    test_data = test_data.shuffle(seed=42).select(range(2048))

    # tokenize the data
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize_function(examples):
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_len)
    tokenized_train_data = train_data.map(tokenize_function, batched=True)
    tokenized_test_data = test_data.map(tokenize_function, batched=True)


    # delete text column
    tokenized_train_data = tokenized_train_data.remove_columns(["text"])
    tokenized_test_data  = tokenized_test_data.remove_columns(["text"])

    # python list -> torch tensor
    tokenized_train_data.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )
    tokenized_test_data.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "label"]
    )

    return tokenized_train_data, tokenized_test_data

In [2]:
import torch

def validation_step(model, validation_loader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for batch in validation_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)  # shape: (batch,)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss               # scalar
            logits = outputs.logits           # shape: (batch, num_labels)

            total_loss += loss.item() * input_ids.size(0)
            preds = torch.argmax(logits, dim=-1)
            total_correct += (preds == labels).sum().item()
            total_count += input_ids.size(0)

    avg_loss = total_loss / total_count
    avg_acc = total_correct / total_count
    return  avg_loss, avg_acc

In [7]:
from xml.parsers.expat import model

import torch
import time
import wandb
from tqdm import tqdm


def train(model, train_dataset, validation_dataset, num_epochs, batch_size, learning_rate, num_workers, optimizer_name = "AdamW", compile_mode = False):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")
    
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    validation_loader = torch.utils.data.DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    if compile_mode:
        model = torch.compile(model) 

    initial_val_loss, initial_val_acc = validation_step(model, validation_loader, device)
    print(f'Initial validation loss: {initial_val_loss}, accuracy: {initial_val_acc}')
    # wandb.log({'val_loss': initial_val_loss, 'val_acc': initial_val_acc})

    for epoch in tqdm(list(range(num_epochs))):
        model.train()

        train_losses = 0.0
        total_correct = 0
        data_time = 0.0
        compute_time = 0.0

        end_time = time.time()

        # 5c) Training loop
        for batch in train_loader:
            ############################################################
            # STUDENT IMPLEMENTATION START
            # 1. clear the gradients stored in optimizer.
            # 2. compute the loss by calling compute_loss
            # 3. backpropagate the loss
            # 4. call optimizer to update the model parameters
            ############################################################

            # data time
            data_time += time.time() - end_time
            compute_start_time = time.time()
            
            # clear gradients
            optimizer.zero_grad()

            # compute loss
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss               # scalar
            logits = outputs.logits           # shape: (batch, num_labels)

            # backpropagate
            loss.backward()

            # update model parameters
            optimizer.step()
            torch.cuda.synchronize()


            # record training loss and accuracy
            end_time = time.time()
            compute_time += end_time - compute_start_time
            train_losses += loss.item() * input_ids.size(0)
            preds = torch.argmax(logits, dim=-1)
            total_correct += (preds == labels).sum().item()

        
        # validation step
        test_loss, test_acc = validation_step(model, validation_loader, device)
        avg_train_loss = train_losses / len(train_dataset)
        train_acc = total_correct / len(train_dataset)

        wandb.log({
            'train/loss': avg_train_loss,
            'train/acc': train_acc,
            'test/loss': test_loss,
            'test/acc': test_acc,
            'time/data_loading': data_time,
            'time/compute': compute_time,
            'time/epoch': data_time + compute_time
        })
        print(f'Epoch {epoch}: train loss: {avg_train_loss}, data loading time: {data_time:.2f}s, compute time: {compute_time:.2f}s')

    return

In [16]:
import torch
import time
import wandb
from tqdm import tqdm


def count_trainable_params(model: torch.nn.Module):
    trainable_tensors = 0
    trainable_elems = 0
    for p in model.parameters():
        if p.requires_grad:
            trainable_tensors += 1
            trainable_elems += p.numel()
    return trainable_tensors, trainable_elems


def count_params_with_grads(model: torch.nn.Module):
    grad_tensors = 0
    grad_elems = 0
    for p in model.parameters():
        if p.grad is not None:
            grad_tensors += 1
            grad_elems += p.grad.numel()
    return grad_tensors, grad_elems


def train_and_count_paramters(
    model,
    train_dataset,
    validation_dataset,
    num_epochs,
    batch_size,
    learning_rate,
    num_workers,
    optimizer_name="AdamW",
    compile_mode=False,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Optimizer
    if optimizer_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    elif optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
    )
    validation_loader = torch.utils.data.DataLoader(
        validation_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )

    if compile_mode:
        model = torch.compile(model)

    # ---- Count trainable params (once) ----
    trainable_tensors, trainable_elems = count_trainable_params(model)
    print(f"Trainable parameters: {trainable_elems:,} (across {trainable_tensors} tensors)")
    wandb.log({
        "model/trainable_param_tensors": trainable_tensors,
        "model/trainable_param_elems": trainable_elems,
    })

    # initial_val_loss, initial_val_acc = validation_step(model, validation_loader, device)
    # print(f"Initial validation loss: {initial_val_loss}, accuracy: {initial_val_acc}")

    for epoch in tqdm(list(range(num_epochs))):
        model.train()

        train_losses = 0.0
        total_correct = 0
        data_time = 0.0
        compute_time = 0.0

        end_time = time.time()

        # Training loop
        for batch_idx, batch in enumerate(train_loader):
            # data time
            data_time += time.time() - end_time
            compute_start_time = time.time()

            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )

            loss = outputs.loss
            logits = outputs.logits

            loss.backward()

            # ---- Count params with grads (after epoch) ----
            if epoch == 0 and batch_idx == 0:
                grad_tensors, grad_elems = count_params_with_grads(model)
                print(f"Parameters with gradients (after one backward): "f"{grad_elems:,} (across {grad_tensors} tensors)")

            optimizer.step()

            if device.type == "cuda":
                torch.cuda.synchronize()

            # record training loss and accuracy
            end_time = time.time()
            compute_time += end_time - compute_start_time
            train_losses += loss.item() * input_ids.size(0)
            preds = torch.argmax(logits, dim=-1)
            total_correct += (preds == labels).sum().item()


        # validation step
        test_loss, test_acc = validation_step(model, validation_loader, device)
        avg_train_loss = train_losses / len(train_dataset)
        train_acc = total_correct / len(train_dataset)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": train_acc,
            "test/loss": test_loss,
            "test/acc": test_acc,
            "time/data_loading": data_time,
            "time/compute": compute_time,
            "time/epoch": data_time + compute_time,
            "model/grad_param_tensors": grad_tensors,
            "model/grad_param_elems": grad_elems,
        })

        print(
            f"Epoch {epoch}: train loss: {avg_train_loss}, "
            f"data loading time: {data_time:.2f}s, compute time: {compute_time:.2f}s, "
            f"params w/ grads: {grad_elems:,} (across {grad_tensors} tensors)"
        )

    return

In [3]:
from csv import writer
import os
import time
import torch
import wandb
from tqdm import tqdm
from torch.profiler import profile, ProfilerActivity, tensorboard_trace_handler
from torch.utils.tensorboard import SummaryWriter

def train_profiler(
    model, train_dataset, validation_dataset,
    num_epochs, batch_size, learning_rate, num_workers,
    profiler_dir="/content/drive/MyDrive/tb_profiler"  # <-- save to Google Drive
):
    # Mount Google Drive (Colab)
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive"):
            os.makedirs("/content/drive", exist_ok=True)
        # If not mounted yet, this will prompt once
        if not os.path.exists("/content/drive/MyDrive"):
            drive.mount("/content/drive")
    except Exception as e:
        print(f"[Warning] Drive mount skipped/failed: {e}")
        return

    # Select device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Initialize optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    # Create DataLoaders
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
    )

    validation_loader = torch.utils.data.DataLoader(
        validation_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=(num_workers > 0),
    )

    # Initial validation before training
    initial_val_loss, initial_val_acc = validation_step(model, validation_loader, device)
    print(f"Initial validation loss: {initial_val_loss}, accuracy: {initial_val_acc}")

    # Make sure the profiler directory exists on Drive
    os.makedirs(profiler_dir, exist_ok=True)

    for epoch in tqdm(range(num_epochs)):
        model.train()

        train_losses = 0.0
        total_correct = 0
        
        data_time = 0.0
        compute_time = 0.0

        end_time = time.time()

        # Save trace files under:
        # /content/drive/MyDrive/tb_profiler/workers_{num_workers}/epoch_{epoch}/
        trace_dir = os.path.join(profiler_dir, f"workers_{num_workers}", f"epoch_{epoch}")
        os.makedirs(trace_dir, exist_ok=True)
        
        # Profile the entire epoch
        with profile(
            schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
            on_trace_ready=tensorboard_trace_handler(trace_dir),
            record_shapes=True,
            with_stack=True,
        ) as prof:

            for step, batch in enumerate(train_loader):
                prof.step()
                # Measure data loading time
                data_time += time.time() - end_time
                compute_start_time = time.time()

                optimizer.zero_grad()

                # Move batch to device
                input_ids = batch["input_ids"].to(device, non_blocking=True)
                attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                labels = batch["label"].to(device, non_blocking=True)

                # Forward pass
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )

                loss = outputs.loss
                logits = outputs.logits

                # Backward pass and parameter update
                loss.backward()
                optimizer.step()

                # Synchronize to get more accurate compute timing
                if torch.cuda.is_available():
                    torch.cuda.synchronize()

                end_time = time.time()
                compute_time += end_time - compute_start_time

                # Track metrics
                train_losses += loss.item() * input_ids.size(0)
                preds = torch.argmax(logits, dim=-1)
                total_correct += (preds == labels).sum().item()

        # Validation step after each epoch
        test_loss, test_acc = validation_step(model, validation_loader, device)

        avg_train_loss = train_losses / len(train_dataset)
        train_acc = total_correct / len(train_dataset)

        wandb.log({
            "train/loss": avg_train_loss,
            "train/acc": train_acc,
            "test/loss": test_loss,
            "test/acc": test_acc,
            "time/data_loading": data_time,
            "time/compute": compute_time,
            "time/epoch": data_time + compute_time
        })

        print(
            f"Epoch {epoch}: train loss {avg_train_loss:.4f}, "
            f"data {data_time:.2f}s, compute {compute_time:.2f}s"
        )
        print(f"[Profiler] Trace saved to: {trace_dir}")

In [ ]:
#C1 & C2
import wandb
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
max_len = 256
batch_size = 32
num_epochs = 5
learning_rate = 1e-4
optimizer = "AdamW"
num_workers = 2
compile_mode = False


wandb.init(project="hpml-hw2-llm")
wandb.config.update({
    "model_name": model_name,
    "max_len": max_len,
    "batch_size": batch_size,
    "lr": learning_rate,
    "optimizer": optimizer,
    "num_workers": num_workers,
    "epochs": num_epochs,
    "compile_mode": compile_mode
})

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  
)

train_dataset, validation_dataset = load_data_and_tokenize()

train(model, train_dataset, validation_dataset)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/2048 [00:00<?, ? examples/s]

Initial validation loss: 0.6935234414413571, accuracy: 0.49755859375


 20%|██        | 1/5 [00:59<03:58, 59.59s/it]

...

Epoch 2: train loss: 0.10688533589564031, data loading time: 0.18s, compute time: 44.08s


 80%|████████  | 4/5 [03:56<00:59, 59.15s/it]

Epoch 3: train loss: 0.07262088460993255, data loading time: 0.17s, compute time: 44.21s


100%|██████████| 5/5 [04:56<00:00, 59.20s/it]

Epoch 4: train loss: 0.05211467490516952, data loading time: 0.18s, compute time: 44.27s


In [19]:
# C3
from transformers import AutoModelForSequenceClassification
import wandb

model_name = "distilbert-base-uncased"
max_len = 256
batch_size = 32
num_epochs = 5
learning_rate = 1e-4
optimizer_name = "AdamW"
compile_mode = False

worker_list = [0, 2, 4, 8]

for num_workers in worker_list:

    wandb.init(
        project="hpml-hw2-llm",
        name=f"workers_{num_workers}",
        reinit=True
    )

    wandb.config.update({
        "model_name": model_name,
        "max_len": max_len,
        "batch_size": batch_size,
        "lr": learning_rate,
        "optimizer": optimizer_name,
        "num_workers": num_workers,
        "epochs": num_epochs,
        "compile_mode": compile_mode
    })

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    train_dataset, validation_dataset = load_data_and_tokenize()

    train(
        model,
        train_dataset,
        validation_dataset,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        num_workers=num_workers
    )

    wandb.finish()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Initial validation loss: 0.693833751603961, accuracy: 0.4931640625


 20%|██        | 1/5 [00:58<03:52, 58.05s/it]

Epoch 0: train loss: 0.5337188690900803, data loading time: 0.17s, compute time: 42.76s


...

 80%|████████  | 4/5 [04:00<01:00, 60.01s/it]

Epoch 3: train loss: 0.07108737679300248, data loading time: 0.44s, compute time: 44.29s


100%|██████████| 5/5 [05:00<00:00, 60.05s/it]

Epoch 4: train loss: 0.036062204595509684, data loading time: 0.45s, compute time: 44.25s


test/acc,█▁█▇█
test/loss,▁▆▄▇█
time/compute,█▃▁▃▃
time/data_loading,█▁▁▂▃
time/epoch,█▃▁▃▃
train/acc,▁▆▇▇█
train/loss,█▄▂▂▁
test/acc,0.8667
test/loss,0.49415
time/compute,44.2549
time/data_loading,0.45051


In [27]:
!ls /content/drive/MyDrive/

In [20]:
# C5: sweep batch size and learning rate (grid)
from transformers import AutoModelForSequenceClassification
import wandb
import torch

model_name = "distilbert-base-uncased"
max_len = 256
num_epochs = 5
optimizer_name = "AdamW"
compile_mode = False
num_workers = 2  

batch_size_list = [16, 32, 64]
lr_list = [5e-5, 1e-4, 5e-4]

for batch_size in batch_size_list:
    for learning_rate in lr_list:

        run_name = f"bs{batch_size}_lr{learning_rate:g}"

        wandb.init(
            project="hpml-hw2-llm",
            group="C5_bs_lr",
            name=run_name,
            reinit=True
        )

        wandb.config.update({
            "exp": "C5",
            "model_name": model_name,
            "max_len": max_len,
            "batch_size": batch_size,
            "lr": learning_rate,
            "optimizer": optimizer_name,
            "num_workers": num_workers,
            "epochs": num_epochs,
            "compile_mode": compile_mode
        })

        # (optional) reduce cross-run GPU memory fragmentation
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )

        train_dataset, validation_dataset = load_data_and_tokenize()

        train(
            model,
            train_dataset,
            validation_dataset,
            num_epochs=num_epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            num_workers=num_workers
        )

        wandb.finish()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Initial validation loss: 0.6941632693633437, accuracy: 0.498046875


 20%|██        | 1/5 [00:59<03:58, 59.72s/it]

Epoch 0: train loss: 0.4265438426518813, data loading time: 0.39s, compute time: 44.77s


...

 80%|████████  | 4/5 [03:53<00:58, 58.29s/it]

Epoch 3: train loss: 0.6932004615664482, data loading time: 0.16s, compute time: 43.79s


100%|██████████| 5/5 [04:51<00:00, 58.30s/it]

Epoch 4: train loss: 0.6958597395569086, data loading time: 0.17s, compute time: 43.57s


test/acc,██▁██
test/loss,█▂▁▁▁
time/compute,█▂▁▃▁
time/data_loading,▅▁█▄▆
time/epoch,█▂▁▃▁
train/acc,█▃▅▄▁
train/loss,█▇▂▁▂
test/acc,0.50342
test/loss,0.6938
time/compute,43.57373
time/data_loading,0.16821


In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sun Mar  1 01:08:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# C6 - Optimizer experiment
import wandb
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
max_len = 256
batch_size = 32
num_epochs = 5
learning_rate = 1e-4
num_workers = 2
compile_mode = False
optimizer_list = ["SGD", "Adam"]

for optimizer_name in optimizer_list:
    print(f"\n\n=== Running with optimizer: {optimizer_name} ===")

    wandb.init(
        project="hpml-hw2-llm",
        group="C6",
        name=f"optimizer_{optimizer_name}",
        reinit=True
    )

    wandb.config.update({
        "model_name": model_name,
        "max_len": max_len,
        "batch_size": batch_size,
        "lr": learning_rate,
        "optimizer": optimizer_name,
        "num_workers": num_workers,
        "epochs": num_epochs,
        "compile_mode": compile_mode
    })

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    train_dataset, validation_dataset = load_data_and_tokenize()

    train_and_count_paramters(
        model,
        train_dataset,
        validation_dataset,
        num_epochs,
        batch_size,
        learning_rate,
        num_workers,
        optimizer_name,
        compile_mode
    )

    wandb.finish()



=== Running with optimizer: SGD ===


model/trainable_param_elems,▁
model/trainable_param_tensors,▁
model/trainable_param_elems,66955010
model/trainable_param_tensors,104


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 66,955,010 (across 104 tensors)


...

  0%|          | 0/1 [00:00<?, ?it/s]

Parameters with gradients (after one backward): 66,955,010 (across 104 tensors)


100%|██████████| 1/1 [00:57<00:00, 57.63s/it]

Epoch 0: train loss: 0.44779287721030414, data loading time: 0.31s, compute time: 43.08s, params w/ grads: 66,955,010 (across 104 tensors)


model/grad_param_elems,▁
model/grad_param_tensors,▁
model/trainable_param_elems,▁
model/trainable_param_tensors,▁
test/acc,▁
test/loss,▁
time/compute,▁
time/data_loading,▁
time/epoch,▁
train/acc,▁
+1,...


In [9]:
# C7 - Compile mode experiment
import wandb
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"
max_len = 256
batch_size = 32
num_epochs = 10
learning_rate = 1e-4
num_workers = 0
optimizer_name = "AdamW"

compile_mode_list = [False, True]

for compile_mode in compile_mode_list:

    wandb.init(
        project="hpml-hw2-llm",
        group="C7",
        name=f"compile_{compile_mode}",
        reinit=True
    )

    wandb.config.update({
        "model_name": model_name,
        "max_len": max_len,
        "batch_size": batch_size,
        "lr": learning_rate,
        "optimizer": optimizer_name,
        "num_workers": num_workers,
        "epochs": num_epochs,
        "compile_mode": compile_mode
    })

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    # 关键：根据 compile_mode 决定是否 torch.compile
    if compile_mode:
        import torch
        model = torch.compile(model)

    train_dataset, validation_dataset = load_data_and_tokenize()

    train(
        model,
        train_dataset,
        validation_dataset,
        num_epochs,
        batch_size,
        learning_rate,
        num_workers,
        optimizer_name
    )

    wandb.finish()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Initial validation loss: 0.690304558724165, accuracy: 0.541015625


 10%|█         | 1/10 [00:56<08:29, 56.61s/it]

Epoch 0: train loss: 0.6031508673913777, data loading time: 0.18s, compute time: 42.06s


...

 90%|█████████ | 9/10 [08:50<00:56, 56.73s/it]

Epoch 8: train loss: 0.043821174102049554, data loading time: 0.17s, compute time: 42.24s


100%|██████████| 10/10 [09:46<00:00, 58.65s/it]

Epoch 9: train loss: 0.028844621952885063, data loading time: 0.18s, compute time: 42.38s


test/acc,▆█▇▄▁▇▄▁▇▇
test/loss,▁▁▂▄▆▄▆█▃▂
time/compute,█▁▁▁▁▁▁▁▁▁
time/data_loading,▂▃▄▂▃▃▄▁▁█
time/epoch,█▁▁▁▁▁▁▁▁▁
train/acc,▁▆▇▇██████
train/loss,█▄▃▂▁▁▁▁▁▁
test/acc,0.86572
test/loss,0.39936
time/compute,42.37622
time/data_loading,0.18445


# HPML HW2 REPORT

---
## Environment Summary
- GPU: Tesla T4, 15360 MiB, 580.82.07
- OS: Linux 6.6.113+ (#1 SMP Mon Feb  2 12:27:57 UTC 2026)
- Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
- PyTorch: 2.10.0+cu128 (CUDA: 12.8, cuDNN: 91002)
- Transformers: 5.0.0
- Datasets: 4.0.0
- W&B: 0.25.0

--- 
## WanDB 
Project Link: https://wandb.ai/desolate0602-columbia-university/hpml-hw2-llm/

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/wandb_overview.png" width="80%" />
</div>

---

## C1 Fine-tuning a Small LLM

### Training Hyperparameters
- Batch Size: 32
- Number of Epochs: 5
- Learning Rate: 1e-4
- Optimizer: AdamW
- Compile Mode: False

### Results

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C1-Training_Loss_Curve.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C1-Training_Accuracy_Curve.png" width="45%" />
</div>

<p align="center">
Loss Curve (left) and Accuracy Curve (right)
</p>

---
## C2 Baseline Timing

### Training Hyperparameters
- Batch Size: 32
- Number of Epochs: 5
- Learning Rate: 1e-4
- Optimizer: AdamW
- Compile Mode: False

### Results

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C2_time_epoch.png" width="32%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C2_time_data.png" width="32%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C2_time_compute.png" width="32%" />
</div>

<div align="center">
<b>Figure</b> Epoch time (left), data loading time (middle), and compute time (right) for C2.
</div>

---
## C3: Effect of DataLoader Workers

### Experiment Configuration

| Category | Parameter | Value |
|----------|-----------|-------|
| Model | Model Name | distilbert-base-uncased |
| Model | Max Length | 256 |
| Training | Batch Size | 32 |
| Training | Epochs | 5 |
| Training | Learning Rate | 1e-4 |
| Training | Optimizer | AdamW |
| System | Compile Mode | False |
| System | DataLoader Workers | 0, 2, 4, 8 |
| Dataset | Train Data Size | 2048 |
| Dataset | Test Data Size | 2048 |

### Performance Results

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C3_time_epoch.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C3_time_data.png" width="45%" />
</div>

<div align="center">
<b>Figure:</b> Epoch time (left) and data loading time (right) under different numbers of DataLoader workers.
</div>

### Summary

1. Since the dataset size is relatively small and samples are already pre-tokenized, increasing the number of DataLoader workers introduces additional multiprocessing overhead (process creation, inter-process communication), which leads to longer data loading time instead of speedup.

2. As the overall training time is dominated by GPU computation rather than data loading, changes in the number of workers do not significantly affect the total epoch time.

---

## C4: Pytorch Profiler

### Experiment Configuration

| Parameter       | Value                     |
|-----------------|---------------------------|
| Model Name      | distilbert-base-uncased   |
| Max Length      | 256                       |
| Batch Size      | 32                        |
| Number of Epochs| 1                         |
| Learning Rate   | 1e-4                      |
| Optimizer       | AdamW                     |
| Num Workers     | 0,1                       |
| Compile Mode    | False                     |
| Train Data Size | 2048                      |
| Train Data Size | 2048                      |

### TensorBoard Report (0 Worker)

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/c4_summary_worker_0.png" width="80%" />
</div>

<div align="center">
<b>Figure:</b> TensorBoard report, num_works = 0
</div>

### TensorBoard Report (1 Worker)

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/c4_summary_worker_1.png" width="80%" />
</div>

<div align="center">
<b>Figure:</b> TensorBoard report, num_works = 1
</div>

### Comparison Table
<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/c4_time_worker_0.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/c4_time_worker_1.png" width="45%" />
</div>

<div align="center">
<b>Figure: </b>Comparison between 0 worker (left) and 1 worker (right)
</div>

### Performance Analysis

1. Since the dataset is relatively small and has already been tokenized, data loading introduces negligible overhead and has minimal impact on overall training time.

2. Compared to data loading, the vast majority of execution time is spent on GPU computation, accounting for more than 99% of the total runtime.

3. The primary bottleneck lies in computational throughput rather than in data input or preprocessing.

---
## C5: Effect of Batch Size and Learning Rate

### Experiment Configuration

| Category | Parameter | Value |
|----------|-----------|-------|
| Model | Model Name | distilbert-base-uncased |
| Model | Max Length | 256 |
| Training | Epochs | 5 |
| Training | Optimizer | AdamW |
| System | Compile Mode | False |
| System | DataLoader Workers | 2 |
| Dataset | Train Data Size | 2048 |
| Dataset | Test Data Size | 2048 |

### Hyperparameter Sweep

- Batch Size: 16, 32, 64  
- Learning Rate: 5e-5, 1e-4, 5e-4  


### Performance Results

#### Time Analysis

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C5_time.png" width="45%" />
</div>

<div align="center">
<b>Figure 1:</b> Training time under different batch size and learning rate settings.
</div>


#### Loss Curves

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C5_train_loss.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C5_test_loss.png" width="45%" />
</div>

<div align="center">
<b>Figure 2:</b> Training loss (left) and test loss (right) under different hyperparameter settings.
</div>


#### Training Accuracy

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C5_train_accuracy.png" width="45%" />
</div>

<div align="center">
<b>Figure 3:</b> Training accuracy under different hyperparameter settings.
</div>

### Summary and Analysis

From the results, we observe that increasing the batch size and learning rate generally improves training speed, as larger batches increase computational throughput and higher learning rates accelerate parameter updates. 

However, this speed improvement comes at the cost of model performance and stability. Larger batch sizes and higher learning rates lead to noisier optimization dynamics, resulting in reduced accuracy and less stable convergence behavior, as reflected in the training and test loss curves.

---

## C6: Optimizer Comparison

### Experiment Configuration

| Category | Parameter | Value |
|----------|-----------|-------|
| Model | Model Name | distilbert-base-uncased |
| Model | Max Length | 256 |
| Training | Epochs | 5 |
| Training | Batch Size | 32 |
| Training | Learning Rate | 1e-4 |
| System | Compile Mode | False |
| System | DataLoader Workers | 2 |
| Dataset | Train Data Size | 2048 |
| Dataset | Test Data Size | 2048 |

### Hyperparameter

- Optimizer: Adam, SGD, AdamW


### Performance Results

#### Time Analysis

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C6_time_epoch.png" width="45%" />
</div>

<div align="center">
<b>Figure 1:</b> Training time under different Optimizer.
</div>


#### Loss Curves

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C6_train_loss.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C6_test_loss.png" width="45%" />
</div>

<div align="center">
<b>Figure 2:</b> Training loss (left) and test loss (right) under different hyperparameter settings.
</div>


#### Training Accuracy

<div align="center">
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C6_train_accuracy.png" width="45%" />
  <img src="https://raw.githubusercontent.com/jasonlee-1024/image-host/main/hw2/C6_test_accuracy.png" width="45%" />
</div>

<div align="center">
<b>Figure 3:</b> Training accuracy under different hyperparameter settings.
</div>

---

## C7: Compiler Mode

### Experiment Configuration

| Category | Parameter | Value |
|----------|-----------|-------|
| Model | Model Name | distilbert-base-uncased |
| Model | Max Length | 256 |
| Training | Epochs | 10 |
| Training | Batch Size | 32 |
| Training | Learning Rate | 1e-4 |
| Training | Optimizer | AdamW |
| System | Compile Mode | False |
| System | DataLoader Workers | 2 |
| Dataset | Train Data Size | 2048 |
| Dataset | Test Data Size | 2048 |

### Hyperparameter

- Compile Mode: False, True

### Results

| Mode                | First Epoch Time (s) | Avg. Time Epochs 6–10 (s) |
|---------------------|----------------------|----------------------------|
| Eager               |       42.2           |       42.8                 |
| Compile (Inductor)  |       66.1           |       42.4                 |

---


# Short Answer Question

#### Q1. What is the input dimension of DistilBERT’s embedding layer?

The input dimension of DistilBERT’s embedding layer is the vocabulary size.

#### Q2. What is the output dimension of the classifier head for IMDB?
It is a binary classification task, so the classifier head outputs logits of shape (batch_size, 2).

#### Q3. Report the number of trainable parameters and parameters with gradients after a backward pass. Include the code you used to compute both.

Trainable parameters: 66,955,010 (across 104 tensors)
Parameters with gradients (after one backward): 66,955,010 (across 104 tensors)

```python
def count_trainable_params(model: torch.nn.Module):
    trainable_tensors = 0
    trainable_elems = 0
    for p in model.parameters():
        if p.requires_grad:
            trainable_tensors += 1
            trainable_elems += p.numel()
    return trainable_tensors, trainable_elems


def count_params_with_grads(model: torch.nn.Module):
    grad_tensors = 0
    grad_elems = 0
    for p in model.parameters():
        if p.grad is not None:
            grad_tensors += 1
            grad_elems += p.grad.numel()
    return grad_tensors, grad_elems
```
#### Q4. How (if at all) does parameter count change when switching from SGD to Adam?

No change

### Q5. Why is the first epoch slower with torch.compile, and why are later epochs typically faster?
The first epoch is slower with torch.compile because it must capture, optimize, and compile the computation graph during the initial execution, while later epochs reuse the optimized graph and therefore run faster with reduced overhead and improved kernel efficiency.